<a href="https://colab.research.google.com/github/anyahuang66/ds2002-fa26/blob/main/2026_10_16_walmart_warm_up_sku_mess_lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Walmart Warm-Up SKU Mess

**Lab — 2026-10-16 · Fall 2026**  

---

## Lab 08 — Walmart Warm-Up (SKU Mess)

This lab uses the **real midterm data** in `../06-midterm-walmart/data/`. It's a warm-up so you hit the ground running on the project. On Kaggle/Colab, upload the data folder or adjust the path.

*If the file isn't found, the cell falls back to a small inline sample so the notebook still runs.*

In [ ]:
import pandas as pd
PATH = '../06-midterm-walmart/data/product_catalog.csv'
try:
    cat = pd.read_csv(PATH)
except FileNotFoundError:
    from io import StringIO
    cat = pd.read_csv(StringIO('sku,product_name,category\nPT-12,pop-tarts strawberry,Breakfast Foods\nSKU#459812,Strawberry Pop-Tart,Breakfast\nPOPTART-STRAW,Pop-Tarts Strawberry,Breakfast'))
cat.head()

,sku,product_name,category
0,PT-12,pop-tarts strawberry,Breakfast Foods
1,SKU#459812,Strawberry Pop-Tart,Breakfast
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast


### Q1 — How many rows look like Pop-Tart products?

Filter on `product_name` containing `pop`, case-insensitively. Print the count and the matching rows.

In [ ]:
# TODO (hint: .str.lower().str.contains('pop'))
pop_mask = cat["product_name"].str.lower().str.contains("pop")
pop_tarts = cat[pop_mask]

print(len(pop_tarts))
display(pop_tarts)

3


,sku,product_name,category
0,PT-12,pop-tarts strawberry,Breakfast Foods
1,SKU#459812,Strawberry Pop-Tart,Breakfast
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast


### Q2 — Which SKUs and names all mean *strawberry* Pop-Tart?

List them. Then note which Pop-Tart rows are **not** strawberry — those must stay separate, and saying so is part of the answer.

In [ ]:
# TODO
#finds pop-tart-like products
pop_rows = cat[cat['product_name'].str.lower().str.contains('pop', na=False)]

#among those rows, find products whose names contain strawberry
strawberry_rows = pop_rows[pop_rows['product_name'].str.lower().str.contains('strawberry', na=False)]

print("Strawberry Pop-Tart candidates:")
print(strawberry_rows[['sku', 'product_name']])

#find pop-tart like products that aren't strawberry
other_pop_rows = pop_rows[~pop_rows['product_name'].str.lower().str.contains('strawberry', na=False)]

print("\nOther Pop-Tart products to keep separate:")
print(other_pop_rows[['sku', 'product_name']])

Strawberry Pop-Tart candidates:
             sku          product_name
0          PT-12  pop-tarts strawberry
1     SKU#459812   Strawberry Pop-Tart
2  POPTART-STRAW  Pop-Tarts Strawberry

Other Pop-Tart products to keep separate:
Empty DataFrame
Columns: [sku, product_name]
Index: []


**Pop-Tart rows that are a different product:** none in the fallback sample.

### Q3 — Build the mapping

A dict from each variant SKU to one canonical SKU. Then apply it in a way that leaves unmapped SKUs as themselves rather than turning them into `NaN` — that single mistake silently deletes most of the catalog.

In [ ]:
# TODO
#use the first strawberry pop-tart SKU as the representation of all other relevant SKUs
if len(strawberry_rows) > 0:
  canonical_sku = strawberry_rows.iloc[0]['sku']

  #map other strawberry SKU types to canonical one
  sku_mapping = {
      sku: canonical_sku
      for sku in strawberry_rows['sku']
      if sku != canonical_sku
    }
else:
  canonical_sku = None
  sku_mapping = {}

print("Canonical SKU:", canonical_sku)
print("SKU mapping:", sku_mapping)

#apply the mapping
cat['canonical_sku'] = cat['sku'].map(sku_mapping).fillna(cat['sku'])

print("\nOriginal and canonical SKUs:")
print(cat[['sku', 'product_name', 'canonical_sku']])

Canonical SKU: PT-12
SKU mapping: {'SKU#459812': 'PT-12', 'POPTART-STRAW': 'PT-12'}

Original and canonical SKUs:
             sku          product_name canonical_sku
0          PT-12  pop-tarts strawberry         PT-12
1     SKU#459812   Strawberry Pop-Tart         PT-12
2  POPTART-STRAW  Pop-Tarts Strawberry         PT-12


### Q4 — Normalize `category`

Strip and title-case, then show the distinct values before and after with counts. Note any pair that is still two labels for one thing — `Breakfast` and `Breakfast Foods` are not the same string but they are the same category.

In [ ]:
# TODO
#display category counts before normalization
print("Category counts BEFORE normalization:")
print(cat['category'].value_counts(dropna=False))

#strip leading/trailing whitespace and convert to title case
cat['category_normalized'] = (
    cat['category']
    .str.strip()
    .str.lower()
    .replace({
        'breakfast foods': 'breakfast'
    })
    .str.title()
)

#display category counts after normalization
print("\nCategory counts AFTER normalization:")
print(cat['category_normalized'].value_counts(dropna=False))

#compare the distinct category labels
print("\nDistinct categories BEFORE:")
print(sorted(cat['category'].dropna().unique()))

print("\nDistinct categories AFTER:")
print(sorted(cat['category_normalized'].dropna().unique()))

Category counts BEFORE normalization:
category
Breakfast          2
Breakfast Foods    1
Name: count, dtype: int64

Category counts AFTER normalization:
category_normalized
Breakfast    3
Name: count, dtype: int64

Distinct categories BEFORE:
['Breakfast', 'Breakfast Foods']

Distinct categories AFTER:
['Breakfast']


### Q5 — Find the fragmentation you have not spotted yet

Hand-scanning a catalog does not scale. **TODO:** build a candidate list programmatically — normalize each product name (lowercase, strip punctuation, sort the words) and group by that, then show any group holding more than one SKU.

This is the workflow from Monday's lecture, and it is how you will actually approach the full catalog on the midterm.

In [ ]:
# TODO
import re

def normalize_product_name(name):
  #convert name to lowercase
  name = str(name).lower()

  #replace punctuation with spaces
  name = re.sub(r'[^\w\s]', ' ', name)

  #split the name into words and remove extra whitespace
  words = name.split()

  #sort the words so different word orders product the same key
  words = sorted(words)

  #return the normalized name
  return ' '.join(words)

#create a normalized name for every product
cat['normalized_name'] = cat['product_name'].apply(
    normalize_product_name
)

#group products by normalized name
#keep only groups containing more than on distinct SKU
fragmented = (
    cat.groupby('normalized_name')
    .agg(
        sku_count=('sku', 'nunique'),
        skus=('sku', list),
        product_names=('product_name', list)
    )
)
# Keep groups with multiple distinct SKUs
fragmented = fragmented[
    fragmented['sku_count'] > 1
].sort_values('sku_count', ascending=False)

print("Potentially fragmented product groups:")
print(fragmented.to_string())

Potentially fragmented product groups:
                      sku_count                    skus                                 product_names
normalized_name                                                                                      
pop strawberry tarts          2  [PT-12, POPTART-STRAW]  [pop-tarts strawberry, Pop-Tarts Strawberry]


### Q6 — Prove the mapping is safe

**TODO:** assert that applying your mapping produced no nulls in the canonical column and did not change the number of rows. Consolidation renames; it never adds or removes.

In [ ]:
# TODO
assert cat['canonical_sku'].notna().all(), (
    "Error: Some canonical SKUs are missing."
)

# Check that the mapping did not change the number of rows.
assert len(cat['canonical_sku']) == len(cat['sku']), (
    "Error: The number of rows changed."
)

print("Both checks passed!")
print("Original row count:", len(cat))
print("Canonical row count:", len(cat['canonical_sku']))
print("Missing canonical SKUs:", cat['canonical_sku'].isna().sum())

Both checks passed!
Original row count: 3
Canonical row count: 3
Missing canonical SKUs: 0


### Write-up

**a)** Name two product families beyond Pop-Tarts that appear fragmented, and how you would group them.

**b)** Where does consolidation stop being a data-cleaning task and become a business decision? Give one example from this catalog where you genuinely are not sure, and say who you would ask.

a) Bottled water and AA batteries. I would compare the product names and SKUs, identify which entries represent the same product, and map their variant SKUs to a single canonical SKU. I would keep products separate when their flabor, size, or other impactful characteristics are different.
b) Consolidation is a data-cleaning task when different SKUs clearly refer to the same product and the differences are only superficial, such as capitalization, punctuation, or word order. It becomes a business decision when I can't confidently tell if two entries represent the same product or different sellable items. For exmaple, I'm not sure if "PORTABLE CHARGER" and "Portable Phone Charger" are referring to one item or two separate items, and thus I wouldn't automatically merge them, but rather ask the product catalog owner if they should share a canonical SKU.